## Analysing Intra-individual Variance with Athlete Biological Passport
    - To build this for biomarkers: IGF-I and P-III-NP 4 key steps are required:
        1. import + load
        2. select biomarker feature 
        3. impute missing values 
        4. standardise globally 
        5. ABP-style leave - one - out scoring 
        6. score results 
    
        
    

In [ ]:
import pandas as pd 
import numpy as np 
from sklearn.impute import SimpleImputer


In [70]:
merged_df = pd.read_csv("C:/Users/jagmeet/bofa_data/merged_df.csv")
#merged_df = merged_df[merged_df['source'] == 'ATHLETE_REF'].copy()
merged_df.shape


(6262, 18)

In [71]:

features = [ 
    'ln_pnp_orion', 
    'ln_pnp_cis',
    'ln_igf_immuno', 
    'ln_igf_immulite', 
    'ln_pnp_siemens', 
    'ln_igf_ms',
    'ln_igf_ids', 
    'ln_igf_imt', 
    'ln_pnp_initial', 
    'ln_igf_initial',
    'ln_pnp_mean', 
    'ln_igf_mean', 
    'avg_pnp', 
    'avg_igf'
    ]

X = merged_df[features]

imputer = SimpleImputer(strategy='mean')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns = features)
#scaler = StandardScaler()
#X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns = features)
#df_scaled = X_scaled.copy()
X_imputed['id'] = merged_df['id'].values # global scaling  (the .values means that you take the array of column and place it row by row into the one specified.)
print(X_imputed)

      ln_pnp_orion    ln_pnp_cis  ln_igf_immuno  ln_igf_immulite  \
0     1.596666e+00  5.816175e-17  -2.210192e-01    -8.881784e-16   
1     1.002278e+00  5.816175e-17  -3.705622e-01    -8.881784e-16   
2     1.011141e+00  5.816175e-17  -3.665167e-01    -8.881784e-16   
3     9.088235e-01  5.816175e-17  -4.244499e-01    -8.881784e-16   
4     1.039446e+00  5.816175e-17  -3.921770e-01    -8.881784e-16   
...            ...           ...            ...              ...   
6257  1.588327e-16  5.816175e-17   3.588600e-16    -8.881784e-16   
6258  1.588327e-16  5.816175e-17   3.588600e-16    -8.881784e-16   
6259  1.588327e-16  5.816175e-17   3.588600e-16    -8.881784e-16   
6260  1.588327e-16  5.816175e-17   3.588600e-16    -8.881784e-16   
6261  1.588327e-16  5.816175e-17   3.588600e-16    -8.881784e-16   

      ln_pnp_siemens     ln_igf_ms    ln_igf_ids    ln_igf_imt  \
0      -1.662233e-16 -1.768883e-16  9.488768e-16  1.104188e-15   
1      -1.662233e-16 -1.768883e-16  9.488768e-16  1

In [75]:

# imputer for ABP - leave out one sample per athlete this way we can use it for checks 
scores = []
ids = []
for athlete in X_imputed['id'].unique(): # get all unique id's from athletes  + process athletes indepndently
    df_a = X_imputed[X_imputed['id'] == athlete].copy() # extrct samples for each unique ID
    
    if len(df_a) < 2: #  need at least 3 samples to compute mean and std
        continue 

    data = df_a[features].values # convert df into np array 
        #Rows = samples
        
        # Columns = biomarkers
    for i in range(len(data)): # loop through each sample individually (since it is now an ARRAY)
        baseline = np.delete(data, i, axis=0)
        current = data[i]
        #current: the sample i am testing + baseline (all other samples for that athlete)

        #baseline	what is “normal” for this athlete + current	the sample you are testing

        mu = baseline.mean(axis=0) # mean for that athelte
        sigma = baseline.std(axis=0) # std for that athlete
        #avoid division by 0
        sigma[sigma == 0] = 1e-6
        z = (current - mu) / sigma
        # score for multivariate deviation 
        
        score = np.sum(z**2)
        scores.append(score)
        ids.append(athlete)


STORE RESULTS   

In [76]:
abp_results = pd.DataFrame({'id': ids, 'abp_score': scores})
abp_results = abp_results[~abp_results['id'].astype(str).str.startswith('RNAG')]
threshold = np.percentile(abp_results['abp_score'], 95)

abp_results['ABP_flag'] = (abp_results['abp_score'] > threshold).astype(int)

abnormal_df = abp_results[abp_results['ABP_flag'] == 1]

print(abnormal_df)



          id     abp_score  ABP_flag
123  8214359  3.892957e+13         1
124  8214359  3.892957e+13         1
